# SAMap: seeded fits vs the 10 Nov fit

In [1]:
import json

import h5py
import pandas as pd
from anndata.io import read_elem
from scipy.stats import pearsonr

NS = "/n/netscratch/eddy_lab/Everyone/nhilgert"
RUNS = {"Nov 10": f"{NS}/2026_11_11_xmens_samap_final",
        "seeded A": f"{NS}/2026_11_12_xmens_samap_seeded_A",
        "seeded B": f"{NS}/2026_11_12_xmens_samap_seeded_B"}


## Joint graph

In [2]:
def joint_graph(run):
    with h5py.File(f"{run}/2_samap/samap_joint.h5ad", "r") as f:
        graph = read_elem(f["obsp/connectivities"]).tocsr()
        names = read_elem(f["obs"]).index
    return graph, names


graphs = {name: joint_graph(run) for name, run in RUNS.items()}
graph_a, names_a = graphs["seeded A"]
for name in ("seeded B", "Nov 10"):
    graph, names = graphs[name]
    order = names.get_indexer(names_a)
    diff = (graph_a - graph[order][:, order]).tocsr()
    diff.eliminate_zeros()
    print(f"seeded A vs {name}: {graph_a.nnz:,} vs {graph.nnz:,} edges; {diff.nnz:,} edge weights differ")

pd.Series({name: json.load(open(f"{run}/2_samap/meta.json"))["mean_alignment"] for name, run in RUNS.items()},
          name="mean alignment")


seeded A vs seeded B: 3,500,203 vs 3,500,203 edges; 0 edge weights differ
seeded A vs Nov 10: 3,500,203 vs 3,498,915 edges; 3,959,746 edge weights differ


Nov 10      0.667561
seeded A    0.673177
seeded B    0.673177
Name: mean alignment, dtype: float64

## Step 4 statistics

In [3]:
def statistics(run):
    s = json.load(open(f"{run}/4_statistics/statistics.json"))
    row = {f"alignment {k}": v for k, v in s["alignment_scores"].items()}
    row.update({f"label agreement, {k}": v["agree_share_of_labelled"] for k, v in s["label_agreement"].items()})
    row.update({k: v["observed"] for k, v in s["ptnull"]["table"].items()})
    row["rho_fibroblast: within-compartment shuffles >= observed (of 1,000)"] = (
        s["ptnull"]["table"]["rho_fibroblast"]["within_compartment_n_ge_obs"])
    return pd.Series(row)


table = pd.DataFrame({name: statistics(run) for name, run in RUNS.items()})
table["A - Nov 10"] = table["seeded A"] - table["Nov 10"]
table.round(4)

,Nov 10,seeded A,seeded B,A - Nov 10
alignment endometrial_fibroblast/endometrial_fibroblast,0.6069,0.6100,0.6100,0.0031
alignment endometrial_fibroblast/decidual_cell,0.1764,0.1781,0.1781,0.0017
alignment decidual_cell/endometrial_fibroblast,0.1701,0.1787,0.1787,0.0086
alignment decidual_cell/decidual_cell,0.5088,0.5089,0.5089,0.0001
"label agreement, all",0.7404,0.7374,0.7374,-0.0030
"label agreement, decidual_cell",0.7353,0.7278,0.7278,-0.0075
"label agreement, endometrial_fibroblast",0.7441,0.7443,0.7443,0.0002
rho_all,0.6660,0.6660,0.6660,0.0000
rho_decidual,0.7340,0.7270,0.7270,-0.0070
rho_fibroblast,0.3050,0.3070,0.3070,0.0020


## Step 3 transfers

In [4]:
def human_transfers(run):
    with h5py.File(f"{run}/3_transfer/Samap_Final.h5ad", "r") as f:
        obs = read_elem(f["obs"])
    return obs[obs["species"].astype(str) == "hs"]


a, nov10 = human_transfers(RUNS["seeded A"]), human_transfers(RUNS["Nov 10"])
nov10 = nov10.reindex(a.index)
pt = pd.DataFrame({"seeded A": a["transferred_mouse_pseudotime"], "Nov 10": nov10["transferred_mouse_pseudotime"]})
pt = pt.astype(float).dropna()
same_call = (a["transferred_mouse_fibdec"].astype(str) == nov10["transferred_mouse_fibdec"].astype(str)).mean()
r = pearsonr(pt["seeded A"], pt["Nov 10"])[0]
print(f"transferred mouse pseudotime, {len(pt):,} human nuclei: Pearson {r:.4f}, "
      f"median |difference| {(pt['seeded A'] - pt['Nov 10']).abs().median():.4f}")
print(f"transferred mouse cell type: same call for {same_call:.2%} of {len(a):,} human nuclei")


transferred mouse pseudotime, 46,353 human nuclei: Pearson 0.9919, median |difference| 0.0099
transferred mouse cell type: same call for 98.09% of 46,818 human nuclei


In [6]:
RUNS["seeded A"]

'/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_12_xmens_samap_seeded_A'

In [7]:
RUNS["seeded B"]

'/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_12_xmens_samap_seeded_B'